In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

In [ ]:
from tools import get_ticket_price, book_flight

In [ ]:
# flights = get_ticket_price("HYD", "IXR", 2, "2026-04-14")

In [ ]:
# flights[0]

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = 'gpt-4.1-mini'
openai = OpenAI()

In [ ]:
from datetime import datetime

today = datetime.today().strftime("%Y-%m-%d (%A)")

In [ ]:
system_message = f"""
Today is {today}.

You are a helpful and courteous airline chatbot assistant which can only be used to search for flights and book air tickets.

You have access to external tools that allow you to search for flights and book flight tickets.

IMPORTANT RULES FOR LOCATION HANDLING:
- The flight search tool requires valid IATA airport codes (e.g., BOM for Mumbai, DEL for Delhi).
- If the user provides a city name (e.g., "Mumbai"), you must internally convert it to the correct IATA code.
- If you are not fully certain about the correct IATA code for a city, DO NOT guess.
- Instead, ask the user a clarifying question (e.g., "Do You mean Mumbai by BOM?").
- Don't ask the user questions like "Could you please confirm the full city names or the airports for "hyd" and "ixr"?"
- Instead, ask "Please confirm the source and destination city: Hyderabad (HYD) to Ranchi (IXR)."
- If multiple airports exist for a city, mention all the options and ask the user to choose.
- Prefer asking the user for IATA codes directly if ambiguity exists.

IMPORTANT RULES FOR DATE HANDLING:
- The flight search tool requires the date in YYYY-MM-DD format.
- If the user provides a date without a year (e.g., "April 10"), assume the nearest future date.
- If the date has already passed in the current year, assume the next year.
- Always convert the final date into YYYY-MM-DD format before calling the tool.
- Do not ask for clarification if the date can be reasonably inferred.
- If you want to put the date in a message for user make sure it is like (e.g. 12th April 2026), and not like (e.g. 12th April or 2026-04-12).
- Instead of something like "Please provide the date for next Friday in YYYY-MM-DD format or confirm if you mean 2026-04-17 for your travel.", just ask "Please confirm your travel date 17th April 2026."

Your responsibility is to:
- Gather all required details (source, destination, number of adults, children ages, and date) before calling the flight search tool.
- Remember, a full ticket is required for children once they turn 2 years old (24 months) on the date of travel.
- So, calculate the number of tickets to be booked accordingly.
- Ask follow-up questions if any required information is missing or when the user response doesn't relate with the query (e.g. for assistant query "No. Of passengers travelling", user response should contain number). Don't ask for all the things in one go.
- Don't ask in this way: "Please confirm the source and destination city: Hyderabad (HYD) to Ranchi (IXR)? Also, please specify the number of adults and children traveling and the ages of any children.".
- Instead, first ask this "Please confirm the source and destination city: Hyderabad (HYD) to Ranchi (IXR)?".
- Then ask this "Please specify the number of adults and children traveling and the ages of any children.".
- Only call the flight search tool when all required parameters are available and valid.
- Show the flight search results in the form of a list.
- If the user has already provided passenger count, do NOT reinterpret numeric inputs as passenger counts. At selection stage, treat numbers as option indices.
- After showing flight prices, if the user wants to proceed with booking, call the `book_flight` tool with the correct parameters.
- Gather all required details (passenger's name and age) from the user, for all the passengers before calling the `book_flight` tool. 
- The flight number, flight company and datetime of journey is available from the users choice in the flight search step.
- After the booking is done, provide the pnr number to the user and thank them for booking with the airline. 
- Also, tell them that they can choose the seats by logging in to www.vimaan.com. Alternately, if no seat is selected, the airline will automatically assign seats.

STRICT BEHAVIOR RULES:
- You are not allowed to cancel or modify any bookings.
- You must not hallucinate or guess airport codes.
- You must provide clear, accurate, and concise responses in no more than one sentence.
- If you are unsure about something, clearly say you do not know.

IMPORTANT RULES FOR RESPONSES:
- Do NOT explain your reasoning or intermediate steps.
- Do NOT say things like "I will check" or "Let me calculate".
- Only provide the final answer directly.

Always maintain a polite and professional tone.
"""

In [ ]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_ticket_price":

            # Parse arguments safely
            try:
                arguments = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                continue

            source_city = arguments.get("source_city")
            destination_city = arguments.get("destination_city")
            adult_count = arguments.get("adult_count")
            children_ages = arguments.get("children_ages", [])
            date_of_journey = arguments.get("date_of_journey")

            # Basic validation (important for robustness)
            if not all([source_city, destination_city, adult_count, date_of_journey]):
                continue

            # Call your function
            price_details = get_ticket_price(
                source_city=source_city,
                destination_city=destination_city,
                adult_count=adult_count,
                date_of_journey=date_of_journey,
                children_ages=children_ages
            )

            # IMPORTANT: content must be string
            responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(price_details)
            })
        elif tool_call.function.name == "book_flight":
            try:
                arguments = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                continue
            passengers = arguments.get("passengers")
            flight_number = arguments.get("flight_number")
            flight_company = arguments.get("flight_company")
            datetime_of_journey = arguments.get("datetime_of_journey")
            pnr_number = book_flight(passengers, flight_number, flight_company, datetime_of_journey)
            responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps({
                    "pnr_number": pnr_number
                })
            })
        
    return responses

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get flight ticket prices given source airport, destination airport, number of adults, children ages, and date of journey.",
            "parameters": {
                "type": "object",
                "properties": {
                    "source_city": {
                        "type": "string",
                        "description": "IATA airport code of departure (e.g., BOM)"
                    },
                    "destination_city": {
                        "type": "string",
                        "description": "IATA airport code of arrival (e.g., DEL)"
                    },
                    "adult_count": {
                        "type": "integer",
                        "description": "Number of adults travelling"
                    },
                    "children_ages": {
                        "type": "array",
                        "items": {
                            "type": "integer"
                        },
                        "description": "List of ages of children travelling (e.g., [5, 10]). Leave empty if no children."
                    },
                    "date_of_journey": {
                        "type": "string",
                        "description": "Date of journey in YYYY-MM-DD format"
                    }
                },
                "required": ["source_city", "destination_city", "adult_count", "date_of_journey"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Book a flight given list of passengers, flight number, flight company, and datetime of journey and return the pnr number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "passengers": {
                        "type": "array",
                        "description": "List of passengers travelling",
                        "items": {
                            "type": "object",
                            "properties": {
                                "first_name": {
                                    "type": "string",
                                    "description": "First name of the passenger"
                                },
                                "middle_name": {
                                    "type": "string",
                                    "description": "Middle name of the passenger (optional)"
                                },
                                "last_name": {
                                    "type": "string",
                                    "description": "Last name of the passenger"
                                },
                                "age": {
                                    "type": "integer",
                                    "description": "Age of the passenger"
                                }
                            },
                            "required": ["first_name", "last_name", "age"]
                        }
                    },
                    "flight_number": {
                        "type": "string",
                        "description": "Flight number"
                    },
                    "flight_company": {
                        "type": "string",
                        "description": "Flight company"
                    },
                    "datetime_of_journey": {
                        "type": "string",
                        "description": "Datetime of journey in YYYY-MM-DD HH:MM:SS format"
                    }
                },
                "required": ["passengers", "flight_number", "flight_company", "datetime_of_journey"]
            }
        }
    }
]

In [ ]:
def chat(message, history):

    messages = [{"role": "system", "content": system_message}]

    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    # Add current user message
    messages.append({"role": "user", "content": message})

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    # Tool calling loop
    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message

        tool_responses = handle_tool_calls(message_obj)

        messages.append(message_obj)
        messages.extend(tool_responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch()